# 类继承关系
```mermaid
classDiagram
    class Records
    class Orders

    %% 继承关系
    Records <|-- Orders
```

# class Orders(Records)
订单类。除了继承 `Records`，还拥有属性 `_close`，用于记录参考价。

## `__init__`
参数
- `wrapper (ArrayWrapper)`：数组包装器对象，包含以下元数据：
  - 行索引（通常是时间戳）
  - 列索引（通常是资产代码）
  - 维度信息和分组设置
- `records_arr (tp.RecordArray)`：NumPy结构化数组，包含所有订单记录，
  - 每条记录包含 id、col、idx、size、price、fees、side 等字段
- `close (tp.Optional[tp.ArrayLike])`：可选的参考价格序列，用于：
  - 绘制价格图表时作为背景
  - 计算订单相对于收盘价的偏差
  - 提供价格上下文信息
- `**kwargs`：传递给父类 `Records` 构造函数的其他参数

```python
    def __init__(self,
                 wrapper: ArrayWrapper,
                 records_arr: tp.RecordArray,
                 close: tp.Optional[tp.ArrayLike] = None,
                 **kwargs) -> None:
        Records.__init__(
            self,
            wrapper,
            records_arr,
            close=close,
            **kwargs
        )
        self._close = close
```

## indexing_func
对 `Orders` 对象执行索引操作。
- 调用 `Records` 的 `indexing_func_meta` 获取索引元数据
- 根据列索引对参考价格数据 `_close` 进行相应的切片
- 创建新的 `Orders` 对象

```python
def indexing_func(self: OrdersT, pd_indexing_func: tp.PandasIndexingFunc, **kwargs) -> OrdersT:
    new_wrapper, new_records_arr, group_idxs, col_idxs = \
        Records.indexing_func_meta(self, pd_indexing_func, **kwargs)
    if self.close is not None:
        new_close = new_wrapper.wrap(to_2d_array(self.close)[:, col_idxs], group_by=False)
    else:
        new_close = None
    return self.replace(
        wrapper=new_wrapper,
        records_arr=new_records_arr,
        close=new_close
    )
```